In [12]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
import operator
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel,Field
load_dotenv()

True

In [13]:
llm = ChatGroq(
    model= "llama-3.3-70b-versatile",
)

In [14]:
class structure(BaseModel):
    feedback:str=Field(description="detail feedback of the essay")
    score:int=Field(description="score out of 10",ge=1,le=10)

In [15]:
class essay_state(TypedDict):
    essay:str
    clarity_of_thoughts:str
    depth_of_analysis:str
    language:str
    score:Annotated[list[int],operator.add]
    avg_score:float
    overall_feedback:str

In [16]:
def cot_evaluate(state:essay_state)->essay_state:
    essay=state['essay']
    prompt=f"evaluate the clarity of thoughts quality of the following essay and provide the score out of 10 \n {essay}"
    structure_output=llm.with_structured_output(structure)
    output=structure_output.invoke(prompt)
    return {"clarity_of_thoughts":output.feedback,"score":[output.score]}

In [17]:
def doa_evaluate(state:essay_state)->essay_state:
    essay=state['essay']
    prompt=f"evaluate the depth of analysis quality of the following essay and provide the score out of 10 \n {essay}"
    structure_output=llm.with_structured_output(structure)
    output=structure_output.invoke(prompt)
    return {"depth_of_analysis":output.feedback,"score":[output.score]}

In [18]:
def final_evaluate(state:essay_state)->essay_state:
    prompt=f"""based on following feedback create a summarize feedback\n language feedback-{state['language']}\n clarity of thoughts feedback-{state['clarity_of_thoughts']}\n
    depth of analysis-{state['depth_of_analysis']}
    """
    overall_feedback=llm.invoke(prompt).content
    
    avg_score=sum(state['score'])/(len(state["score"]))
    return {"overall_feedback":overall_feedback,"avg_score":avg_score}


In [19]:
def lang_evaluate(state:essay_state)->essay_state:
    essay=state['essay']
    prompt=f"evaluate the language quality of the following essay and provide the score out of 10 \n {essay}"
    structure_output=llm.with_structured_output(structure)
    output=structure_output.invoke(prompt)
    
    return {"language":output.feedback,"score":[output.score]}

In [20]:
graph=StateGraph(essay_state)
graph.add_node("evaluate_lang",lang_evaluate)
graph.add_node("evaluate_cot",cot_evaluate)
graph.add_node("evaluate_doa",doa_evaluate)
graph.add_node("evaluate_final",final_evaluate)
graph.add_edge(START, "evaluate_doa")
graph.add_edge(START, "evaluate_cot")
graph.add_edge(START, "evaluate_lang")

graph.add_edge("evaluate_doa", "evaluate_final")
graph.add_edge("evaluate_cot", "evaluate_final")
graph.add_edge("evaluate_lang", "evaluate_final")

graph.add_edge("evaluate_final", END)
workflow=graph.compile()


In [21]:
essay="""
Artificial Intelligence (AI) is becoming one of the most important technologies in the modern world. It is changing the way people work, study, communicate, and solve problems. In Pakistan, AI has the potential to improve many sectors such as education, healthcare, agriculture, business, and security. With a large young population and growing interest in technology, Pakistan can use AI to achieve economic growth and development.

One of the major roles of AI in Pakistan is in the field of education. AI-powered tools can help students learn more effectively through smart tutoring systems, online learning platforms, and personalized study materials. Universities in Pakistan are also introducing AI-related courses to prepare students for future careers. This can help create skilled professionals who can contribute to the technology industry.

In healthcare, AI can assist doctors in diagnosing diseases, analyzing medical reports, and improving patient care. AI systems can help hospitals manage patient records and provide faster medical services, especially in remote areas where healthcare facilities are limited. During emergencies, AI can also help predict disease outbreaks and support better decision-making.

Agriculture is another important sector where AI can play a major role in Pakistan. Farmers can use AI-based technologies to monitor crops, predict weather conditions, and detect plant diseases. This can increase crop production and reduce losses. Since agriculture is a major part of Pakistan’s economy, AI can help improve food security and farmers’ incomes.

AI is also creating opportunities in business and industry. Many companies are using AI for customer service, marketing, data analysis, and automation. Pakistani freelancers and software developers are using AI tools to work with international clients and earn income online. This can strengthen the country’s digital economy and create new job opportunities.

However, Pakistan also faces challenges in adopting AI. Limited technological infrastructure, lack of awareness, and shortage of skilled professionals are some major issues. The government and educational institutions need to invest in research, training, and modern technology to fully benefit from AI. Ethical use of AI and data privacy should also be considered to avoid misuse.

In conclusion, Artificial Intelligence has the power to transform Pakistan in many positive ways. It can improve education, healthcare, agriculture, and business while creating new opportunities for the youth. If Pakistan invests properly in AI development and education, it can become a strong participant in the global technology world and achieve long-term progress.

"""

In [22]:
initial_input = {
    "essay": essay,
}

workflow.invoke(initial_input)

{'essay': '\nArtificial Intelligence (AI) is becoming one of the most important technologies in the modern world. It is changing the way people work, study, communicate, and solve problems. In Pakistan, AI has the potential to improve many sectors such as education, healthcare, agriculture, business, and security. With a large young population and growing interest in technology, Pakistan can use AI to achieve economic growth and development.\n\nOne of the major roles of AI in Pakistan is in the field of education. AI-powered tools can help students learn more effectively through smart tutoring systems, online learning platforms, and personalized study materials. Universities in Pakistan are also introducing AI-related courses to prepare students for future careers. This can help create skilled professionals who can contribute to the technology industry.\n\nIn healthcare, AI can assist doctors in diagnosing diseases, analyzing medical reports, and improving patient care. AI systems can 